# Propositional Logic, Resolution Refutation, and Alpha-Beta Pruning

**Syllabus mapping:** search: adversarial; logic: propositional, predicate.

**Objectives:** construct and evaluate truth tables; determine satisfiability,
validity, and logical entailment; implement Minimax with Alpha-Beta pruning
and trace pruned branch counts.

## Theoretical Foundations

### 1. Propositional Logic
- **Validity (Tautology):** A sentence is valid if it is true in **all** models (e.g. $P \lor \neg P$).
- **Satisfiability:** A sentence is satisfiable if it is true in **at least one** model.
- **Entailment:** $\alpha \models \beta$ iff in every model where $\alpha$ is true, $\beta$ is also true.
- **Proof by Resolution Refutation:**
  $$\alpha \models \beta \iff \alpha \land \neg \beta \text{ is unsatisfiable (derives empty clause } \Box).$$

### 2. Alpha-Beta Pruning in Adversarial Search
Minimax explores all $O(b^d)$ game tree nodes. Alpha-Beta pruning maintains two bounds:
- $\alpha$: The best (highest) value found so far by any choice along the path for MAX. Initialized to $-\infty$.
- $\beta$: The best (lowest) value found so far by any choice along the path for MIN. Initialized to $+\infty$.

**Pruning Condition:**
Whenever $\alpha \ge \beta$, the remaining children of the current node can be pruned because the opponent would never allow play to reach this state.
- **Best-case complexity:** $O(b^{d/2})$, doubling the searchable search depth.

In [ ]:
from itertools import product

# 1. Propositional Logic Truth Table Evaluator
def evaluate_implication(p, q):
    return (not p) or q

def evaluate_biconditional(p, q):
    return p == q

# Test tautology: (P -> Q) or (Q -> P)
models = list(product([True, False], repeat=2))
is_tautology = True
print("Model (P, Q) | (P -> Q) | (Q -> P) | (P -> Q) or (Q -> P)")
print("-" * 55)
for p, q in models:
    p_imp_q = evaluate_implication(p, q)
    q_imp_p = evaluate_implication(q, p)
    result = p_imp_q or q_imp_p
    if not result:
        is_tautology = False
    print(f"{str(p):<5} {str(q):<5} | {str(p_imp_q):<9} | {str(q_imp_p):<9} | {str(result)}")

print(f"\nFormula is a TAUTOLOGY: {is_tautology}")

# 2. Minimax with Alpha-Beta Pruning Implementation
def alphabeta_trace(node, depth, is_max, alpha, beta, path="Root"):
    if isinstance(node, (int, float)):
        print(f"  Leaf {path}: value = {node} [alpha={alpha}, beta={beta}]")
        return node, 0

    pruned_count = 0
    if is_max:
        val = -float('inf')
        for i, child in enumerate(node):
            child_path = f"{path}->C{i+1}"
            child_val, p = alphabeta_trace(child, depth + 1, False, alpha, beta, child_path)
            pruned_count += p
            val = max(val, child_val)
            alpha = max(alpha, val)
            if beta <= alpha:
                remaining = len(node) - (i + 1)
                pruned_count += remaining
                print(f"  ** PRUNED at {path}: beta ({beta}) <= alpha ({alpha}), skipped {remaining} branch(es) **")
                break
        return val, pruned_count
    else:
        val = float('inf')
        for i, child in enumerate(node):
            child_path = f"{path}->C{i+1}"
            child_val, p = alphabeta_trace(child, depth + 1, True, alpha, beta, child_path)
            pruned_count += p
            val = min(val, child_val)
            beta = min(beta, val)
            if beta <= alpha:
                remaining = len(node) - (i + 1)
                pruned_count += remaining
                print(f"  ** PRUNED at {path}: beta ({beta}) <= alpha ({alpha}), skipped {remaining} branch(es) **")
                break
        return val, pruned_count

# Tree: Root (MAX) with two MIN children A: [3, 5], B: [2, 9]
game_tree = [[3, 5], [2, 9]]
print("\n--- Tracing Alpha-Beta Pruning ---")
root_val, pruned = alphabeta_trace(game_tree, 0, True, -float('inf'), float('inf'))
print(f"\nRoot Minimax Value: {root_val}")
print(f"Total Pruned Subtrees/Leaves: {pruned}")

## GATE-Style Practice

**NAT:** Consider a two-player zero-sum game tree with MAX at the root.
The root has two MIN children, $A$ and $B$. Child $A$ has two leaf children
with values $3$ and $5$ (evaluated from left to right). Child $B$ has two
leaf children with values $2$ and $9$ (evaluated from left to right).
Using Alpha-Beta pruning with standard left-to-right evaluation, how many
leaf nodes are **pruned** (not evaluated)?

**MCQ:** Which of the following propositional logic formulas is a **tautology**
(valid in all models)?

A. $(P \to Q) \to P$
B. $(P \to Q) \lor (Q \to P)$
C. $(P \land Q) \to (P \land \neg Q)$
D. $(P \lor Q) \to (P \land Q)$

**MSQ:** Which of the following statements regarding Alpha-Beta pruning are TRUE?

A. Alpha-Beta pruning always returns the exact same minimax value at the root as standard Minimax search.
B. In the best-case move ordering, Alpha-Beta pruning reduces the effective branching factor from $b$ to $\sqrt{b}$.
C. At a MAX node, the value of $\alpha$ can only increase or remain unchanged.
D. If $\alpha \ge \beta$ at any node, searching further children of that node cannot alter the minimax decision of the parent.

## Solutions

NAT: **1**.
1. Child $A$ (MIN node):
   - Left leaf: evaluates to 3. MIN bound becomes $\min(\infty, 3) = 3$.
   - Right leaf: evaluates to 5. MIN bound becomes $\min(3, 5) = 3$.
   - Child $A$ returns value 3 to Root (MAX).
2. Root (MAX node):
   - Sets $\alpha = \max(-\infty, 3) = 3$.
3. Child $B$ (MIN node, with $\alpha = 3, \beta = \infty$):
   - Left leaf: evaluates to 2. MIN bound becomes $\beta = \min(\infty, 2) = 2$.
   - Pruning condition checked: $\beta \le \alpha$ ($2 \le 3$).
   - **Cutoff triggered!** The right leaf (value 9) is pruned.
Total leaves pruned: **1**.

MCQ: **B**.
- $(P \to Q) \lor (Q \to P) \equiv (\neg P \lor Q) \lor (\neg Q \lor P) \equiv (\neg P \lor P) \lor (Q \lor \neg Q) \equiv \text{True} \lor \text{True} \equiv \text{True}$.
This statement is unconditionally true in every boolean model.

MSQ: **A, B, C, D**.
All four statements are foundational theorems in adversarial search:
- A: Pruning is sound and optimal; it discards only branches provably irrelevant to the root decision.
- B: Best-case time complexity is $O(b^{d/2})$, which corresponds to branching factor $\sqrt{b}$.
- C: MAX only pushes the lower bound $\alpha$ upward.
- D: $\alpha \ge \beta$ guarantees the ancestor already has a better or equal guaranteed alternative.